## Raw Memory, Strides & C-Types

The Direct Triton/CUDA Bridge

Triton, CUDA, and C++ kernels do not work with high-level Python objects. They operate directly on raw memory pointers, byte offsets, and stride arithmetic.

## 1. Strides and The General Offset Formula

When a multi-dimensional tensor is flattened into 1D memory, strides define how many elements (or bytes) you must jump in physical memory to move 1 step along each dimension.

### Row-Major (C-Style, Default in Python/PyTorch/C++)

Last dimension is contiguous (stride = 1).

**Formula for any N-D index** $(i_0, i_1, \dots, i_{n-1})$:

$$\text{Memory Offset} = \sum_{k=0}^{n-1} (i_k \times \text{stride}_k)$$

**For a 3D Tensor of shape** $(D, H, W)$:

$$\text{Offset}(d, h, w) = d \times (H \times W) + h \times W + w \times 1$$

In Triton kernels, you write this stride math explicitly to load block pointers from global memory:

```python
tl.load(ptr + offsets_row[:, None] * stride_r + offsets_col[None, :] * stride_c)
```

## 2. Strides = "How Far Do I Jump?"

Suppose:

```
A B C
D E F
```

Memory is actually:

```
[A, B, C, D, E, F]
```

Each element has a position:

```
A=0  B=1  C=2  D=3  E=4  F=5
```

A stride tells you: "If I move 1 step in this dimension, how many memory elements do I jump?"

For this 2×3 array:

```
shape   = (2, 3)
strides = (3, 1)
```

Why?

- Move one row down → jump 3 elements → `stride_row = 3`
- Move one column right → jump 1 element → `stride_col = 1`

So:

```
offset = row * stride_row + col * stride_col
```

**For E:**

```
offset = 1 * 3 + 1 * 1
       = 4
```

So E lives at `buffer[4]`.

## The Important Mental Model

A tensor index:

```python
tensor[r, c]
```

is really just:

```
base_pointer + r*stride_r + c*stride_c
```

That's basically what you'll write in Triton/CUDA.

## 3. Why Transpose Doesn't Need Copying

Normally you might imagine transpose doing:

```
[A B C]       [A D]
[D E F]  →    [B E]
              [C F]
```

But you don't actually need to move anything.

**Original:**

```
shape   = (2, 3)
strides = (3, 1)
```

**Transposed view:**

```
shape   = (3, 2)
strides = (1, 3)
```

Same memory! The buffer is still:

```
[A B C D E F]
```

Only the rules for calculating offsets changed.

So:

```python
transposed[1, 0]
```

means:

```
1 * 1 + 0 * 3 = 1
→ B
```

**This is called a view.**

- View = different way of looking at the same memory.
- That's why modifying the transpose also modifies the original.

## 4. memoryview = Python's Window into Raw Memory

Normally Python hides memory details from you.

```python
x = [1, 2, 3]
```

You don't care where 1, 2, 3 physically live.

`memoryview` gets you closer to the hardware:

```python
raw = bytearray(...)
mv = memoryview(raw)
```

Now you're saying: "Give me access to the underlying buffer without copying it."

And:

```python
mv.cast("i")
```

means roughly: "Interpret every 4 bytes as a C-style 32-bit integer."

So if bytes represent:

```
01 00 00 00 | 02 00 00 00
```

you can see them as:

```
[1, 2]
```

### Important Distinction

`memoryview` doesn't magically convert the data. It reinterprets the same bytes.

That's very important when working with low-level systems.

## 5. ctypes = Python ↔ C Bridge

Now we're one level lower.

```python
import ctypes
```

`ctypes` lets Python talk directly to native C libraries.

For example:

| C | Python ctypes |
|---|---|
| `int` | `c_int` |
| `float` | `c_float` |
| `double` | `c_double` |
| `void*` | `c_void_p` |

So:

```python
ctypes.c_float
```

means: "I want a value that looks like a C float."

## 6. ctypes Pointers

This:

```python
ctypes.POINTER(ctypes.c_float)
```

means: pointer → float

Think:

```
0x7FFA1234
      │
      ▼
   float
```

And:

```python
ctypes.addressof(c_array)
```

gives you the actual memory address, like:

```
0x7f83a21c4000
```

That's the kind of thing CUDA/Triton ultimately operates around.

In [16]:
raw_bytes = bytearray(b"\x01\x00\x00\x00\x02\x00\x00\x00") # 8 bytes
mv = memoryview(raw_bytes).cast("i") # Reinterpret as 32-bit signed integers (4 bytes each)

print(mv[0])  # 1
print(mv[1])  # 2

# Mutating mv directly mutates the raw backing bytearray!
mv[0] = 99
print(raw_bytes)  # bytearray contains the updated raw bytes

1
2
bytearray(b'c\x00\x00\x00\x02\x00\x00\x00')


## Understanding Memory Layout and cast()

### Think of Memory as Boxes

Imagine RAM is a long row of 1-byte boxes:

```
RAM:

[ box ][ box ][ box ][ box ][ box ][ box ][ box ][ box ]
   1     2     3     4     5     6     7     8
```

Each box = 1 byte = 8 bits.

Now suppose I want to store a 32-bit int. A 32-bit int needs:

```
32 bits = 4 bytes
```

So it occupies 4 boxes:

```
[       32-bit integer       ][       another       ]
[ 1 byte ][ 1 ][ 1 ][ 1 ]    [ 1 ][ 1 ][ 1 ][ 1 ]
```

### What `cast("i")` Does

The memory itself doesn't change.

**Before casting**, Python looks at it as:

```
[byte][byte][byte][byte][byte][byte][byte][byte]
```

**After `memoryview(...).cast("i")`**, you're telling Python:

"Hey, treat every 4 bytes together as one integer."

So Python now sees:

```
[int  ][int  ]
 4B      4B
```

Nothing was converted from 32-bit to 48-bit or anything like that. We're simply changing how we interpret the same memory.

## Super Concrete Example

Suppose memory contains:

```
01 00 00 00
```

That's 4 bytes.

If we interpret those 4 bytes as a 32-bit integer, we get:

```
1
```

Now imagine:

```
01 00 00 00 | 02 00 00 00
```

**Raw-byte interpretation:**

```
[01][00][00][00][02][00][00][00]
```

**`cast("i")` interpretation:**

```
[     1     ][     2     ]
   4 bytes      4 bytes
```

### Why Do We Want This?

Because our tensor might be:

```
[1, 2]
```

where each number is a 32-bit int.

We don't want to manually deal with:

```
byte 0
byte 1
byte 2
byte 3
...
```

We want:

```python
buf[0]  # 1
buf[1]  # 2
```

So `cast` tells Python what a "single element" means.

### Key Takeaway

Memory is just bytes. `cast()` tells Python how many bytes should be grouped together and what type they represent.

So:

- `"b"` → 1 byte per element
- `"i"` → 4 bytes per element
- `"f"` → 4 bytes per element
- `"d"` → 8 bytes per element

And 32-bit = 4 bytes, not 48 bits. That's the whole idea.

## Exercise 1: Zero-Copy Strided Tensor Slice Engine

Build a pure Python class `StridedArray2D` without using NumPy.

### Requirements

- **`__init__(self, raw_buffer: bytearray, shape: tuple[int, int], strides: tuple[int, int] = None)`**
  - Store `shape = (rows, cols)`.
  - If `strides` is not provided, compute default row-major element strides: `strides = (cols, 1)`.
  - Store `self.buf = memoryview(raw_buffer).cast("i")` (treating every 4 bytes as a 32-bit int).

- **`__getitem__(self, key: tuple[int, int])`**
  - Unpack `r, c = key`.
  - Compute flat offset using stride formula: `offset = r * self.strides[0] + c * self.strides[1]`.
  - Return `self.buf[offset]`.

- **`transpose(self)`**
  - Returns a new `StridedArray2D` pointing to the exact same memory buffer (`self.buf`), but with:
    - `shape = (cols, rows)`
    - `strides = (self.strides[1], self.strides[0])` (swapped strides!)

### Verification

Modifying an element in the transposed view instantly reflects in the original array because no data was copied!

In [17]:
class StrideArray2D:
    def __init__(
        self,
        raw_buffer: bytearray,
        shape: tuple[int, int],
        strides: tuple[int, int] = None
    ):
        self.row, self.col = shape

        if strides is None:
            self.strides = (self.col, 1)
        else:
            self.strides = strides

        self.buf = memoryview(raw_buffer).cast("i")
        self.raw_buffer = raw_buffer

    def __getitem__(self, key: tuple[int, int]):
        r, c = key

        r_stride, c_stride = self.strides

        offset = r * r_stride + c * c_stride

        return self.buf[offset]

    def __setitem__(self, key: tuple[int, int], value: int):
        r, c = key

        r_stride, c_stride = self.strides

        offset = r * r_stride + c * c_stride

        self.buf[offset] = value

    def transpose(self):
        return StrideArray2D(
            self.raw_buffer,
            shape=(self.col, self.row),
            strides=(self.strides[1], self.strides[0])
        )

In [18]:
# Create raw memory for 6 int32 values
raw = bytearray(24)

# Put values into the buffer
buf = memoryview(raw).cast("i")

values = [1, 2, 3, 4, 5, 6]

for i, value in enumerate(values):
    buf[i] = value


# Create our 2D array
arr = StrideArray2D(
    raw,
    shape=(2, 3)
)

print("Original:")
print(arr[0, 0], arr[0, 1], arr[0, 2])
print(arr[1, 0], arr[1, 1], arr[1, 2])


# Transpose
t = arr.transpose()

print("\nTransposed:")
print(t[0, 0], t[0, 1])
print(t[1, 0], t[1, 1])
print(t[2, 0], t[2, 1])


# Modify transpose
t[0, 1] = 99

print("\nAfter t[0,1] = 99:")

print("Transpose:")
print(t[0, 0], t[0, 1])
print(t[1, 0], t[1, 1])
print(t[2, 0], t[2, 1])

print("\nOriginal:")
print(arr[0, 0], arr[0, 1], arr[0, 2])
print(arr[1, 0], arr[1, 1], arr[1, 2])

Original:
1 2 3
4 5 6

Transposed:
1 4
2 5
3 6

After t[0,1] = 99:
Transpose:
1 99
2 5
3 6

Original:
1 2 3
99 5 6


## Exercise 2: Invoking Native C Functions with ctypes

Write a Python script that uses `ctypes` to interface with standard C library math and memory functions directly.

In [19]:
import ctypes
import ctypes.util

libc = ctypes.CDLL(ctypes.util.find_library("c") or "msvcrt")

### How ctypes Bridges Python to C

Normally Python does:

```
Python code → Python function
```

Here we're doing:

```
Python → C library → C function
```

`libc` is the C standard library. It contains functions like:

- `puts()`
- `malloc()`
- `memcpy()`
- ...

In [20]:
libc.puts.argtypes = [ctypes.c_char_p]
libc.puts.restype = ctypes.c_int

### Setting Function Signatures

You're basically telling ctypes:

"Before calling this C function, know what types we're passing and what type comes back."

In [21]:
libc.puts(b"Direct call to C puts!")

0

### String Encoding

The `b` prefix is important:

```python
b"hello"
```

means **bytes**, which is much closer to what C's `char*` expects.

```
Python
  │
  │ b"hello"
  ▼
char* ──────→ C puts()
                 │
                 ▼
               output
```

## Creating a C Array

"Create a C array type containing 5 floats."

```python
CFloatArray5 = ctypes.c_float * 5
```

In [22]:
CFloatArray5 = ctypes.c_float * 5

In [23]:
c_array = CFloatArray5(
    1.5,
    2.5,
    3.5,
    4.5,
    5.5
)

### Memory Layout

```
c_array
   │
   ▼
┌──────┬──────┬──────┬──────┬──────┐
│ 1.5  │ 2.5  │ 3.5  │ 4.5  │ 5.5  │
└──────┴──────┴──────┴──────┴──────┘
   4B     4B     4B     4B     4B
```

Because `c_float` is normally 32 bits = 4 bytes.

So total memory:

```
5 × 4 bytes = 20 bytes
```

In [24]:
address = ctypes.addressof(c_array)

print(hex(address))

0x1e1bf3ac4b0


### Memory Addresses

That number is basically: "Where does this array start in memory?"

Imagine:

```
address = 0x1000

0x1000 → 1.5
0x1004 → 2.5
0x1008 → 3.5
0x100C → 4.5
0x1010 → 5.5
```

Each address is 4 bytes apart (size of a float).

## Low-Level Memory Thinking

Instead of thinking:

```python
tensor = [1.5, 2.5, 3.5, 4.5, 5.5]
```

low-level code thinks more like:

```
pointer = 0x1000
dtype   = float32
length  = 5
stride  = 1
```

Then to get element `i`:

```
address = pointer + i × sizeof(float)
```

For i = 3:

```
address = 0x1000 + 3 × 4
        = 0x100C
```

→ that's where 4.5 lives.

## Summary: The Three Powers of ctypes

```
ctypes.CDLL()
      ↓
"Python can call native C"

ctypes.c_float * 5
      ↓
"Python can create C-style memory"

ctypes.addressof()
      ↓
"Python can see the actual memory address"
```